# DSA benchmark on Colab

Clone → environment → data → evaluate. Choose **Runtime → Change runtime type → GPU**, then run the cells in order.
Results go to Google Drive, so an interrupted evaluation resumes where it stopped. To share them, copy the
Drive `results/` folder into your local clone's `results/` and commit (see `docs/BENCHMARK.md`).

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/sttruji/dynamic-state-anchoring.git"
REPO_REF = "main"  # A commit SHA reproduces an exact harness version.
REPO_DIR = Path("/content/dynamic-state-anchoring")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# Isolated environment with the pinned transformers; Colab's CUDA PyTorch is reused.
VENV = Path("/content/dsa-venv")
subprocess.run([sys.executable, "scripts/setup_colab.py", "--venv-dir", str(VENV),
                "--requirements", "requirements-colab.txt"], check=True)
PYTHON = str(VENV / "bin/python")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/dynamic-state-anchoring")
os.environ["DSA_DATA_DIR"] = str(DRIVE / "data")        # Built once, reused every session.
os.environ["HF_HOME"] = str(DRIVE / "cache/huggingface")  # Model downloads cached too.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
RESULTS = DRIVE / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

In [ ]:
# About 2 minutes the first time; later sessions only re-verify hashes.
# Add --eval-only to skip the large training corpora when you only need evaluation.
!{PYTHON} scripts/setup_data.py

In [ ]:
# Plug in any model: gpt2-vanilla, gpt2-sft, codi, lotus (needs checkpoint=...), dsa (needs checkpoint=...),
# or your own module:Class. Add --test only for final held-out numbers.
!{PYTHON} scripts/evaluate.py --model gpt2-sft --suite dev --results-dir {RESULTS}

In [ ]:
!{PYTHON} scripts/evaluate.py --table --results-dir {RESULTS}